02 - Train the study-level model
#
# Reads the mosaic corpus from notebook 01, mounted as a kernel source, and
# trains one fold. Each fold runs as its own kernel (rsna-knee-train-fold-0,
# -fold-1, ...), because pushing them all to one id would overwrite earlier
# weights.
#
# Validation reports two numbers: weak AUC over ~880 held-out studies (stable,
# used for checkpoint selection) and gold AUC over the ~12 human-labelled
# studies in the fold (honest but very noisy - the noise floor is ~0.03).

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
!cd {CODE} && python scripts/build_weak_labels.py

In [ ]:
FOLD = 0
BACKBONE = "resnet34"
BATCH = 2
POSW = 0
# The MIL structure feeds MAX_SERIES * SLICES = 64 images per study, so batch 2
# is already 128 images per forward at 384. A heavier backbone needs batch 1 and
# more accumulation rather than a bigger batch -- the multiplier, not the model,
# is what fills the GPU here.
!cd {CODE} && python scripts/train.py --fold {FOLD} --epochs 8 --backbone {BACKBONE} \
    --batch {BATCH} --accum {16 // BATCH} --workers 2 --max-minutes 480 \
    --pos-weight-cap {POSW} --out /kaggle/working/models

In [ ]:
import json, glob
for h in sorted(glob.glob("/kaggle/working/models/*_history.json")):
    print(h)
    print(json.dumps(json.load(open(h)), indent=2))